In [1]:
import pandas as pd
import re
import os
import paramiko
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from itertools import combinations
from pprint import pprint
from itertools import chain
from sqlalchemy import create_engine
ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy()) 
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
nsx_ds_good_dmux = pd.read_csv(r'c:\Users\glj523\OneDrive - University of Copenhagen\data\illumina_analysis\abidmux_qc_AOZCK_full.csv')
nsx_ds_good_dmux['Read Type'] = nsx_ds_good_dmux['suffix'].apply(lambda x: x.split("_")[1] if len(x.split("_")) > 1 else None)
nsx_ds_good_dmux['Lane'] = nsx_ds_good_dmux['suffix'].apply(lambda x: x.split("_")[0])

In [3]:
all_multiqc_data_old = pd.read_csv(r'C:\Users\glj523\OneDrive - University of Copenhagen\data\illumina_analysis\data.csv')
ns6_multiqc_data = pd.read_csv(r'c:\Users\glj523\OneDrive - University of Copenhagen\data\illumina_analysis\all_ns6_multiqc.csv')
ns6_ds_multiqc_all = pd.read_csv(r'c:\Users\glj523\OneDrive - University of Copenhagen\data\illumina_analysis\ns6_ds_multiqc_all.csv')
raw_read_counts_nsx_ds = pd.read_csv(r"c:\Users\glj523\OneDrive - University of Copenhagen\data\illumina_analysis\line_counts.tsv", sep='\t')
raw_read_counts_nsx_ds['File'] = raw_read_counts_nsx_ds['File'].apply(lambda x: x.split('/')[-1])
raw_read_counts_nsx_ds['fastq_id'] = raw_read_counts_nsx_ds['File'].apply(lambda x: x.split('_')[0])
raw_read_counts_nsx_ds = raw_read_counts_nsx_ds[raw_read_counts_nsx_ds['fastq_id'] != 'Undetermined']
raw_read_counts_nsx_ds['eDNA Concentration'] = raw_read_counts_nsx_ds['File'].apply(lambda x: x.split('_')[1])
raw_read_counts_nsx_ds['Read ID'] = raw_read_counts_nsx_ds['File'].apply(lambda x: x.split('_')[2])
raw_read_counts_nsx_ds['Lane'] = raw_read_counts_nsx_ds['File'].apply(lambda x: x.split('_')[3])
raw_read_counts_nsx_ds['Read Type'] = raw_read_counts_nsx_ds['File'].apply(lambda x: x.split('_')[4])
raw_read_counts_nsx_ds['library_id'] = raw_read_counts_nsx_ds['fastq_id'].apply(lambda x: x.split('-')[0])
raw_read_counts_nsx_ds['robot_sample_id'] = raw_read_counts_nsx_ds['fastq_id'].apply(lambda x: x.split('-')[1])
raw_read_counts_nsx_ds['archive_sample_id'] = raw_read_counts_nsx_ds['fastq_id'].apply(lambda x: '-'.join(x.split('-')[2:]))
raw_read_counts_nsx_ds = raw_read_counts_nsx_ds.rename(columns={'LineCount': "fastqc_raw__Total Sequences"})
raw_read_counts_nsx_ds['fastqc_raw__Total Sequences'] = raw_read_counts_nsx_ds['fastqc_raw__Total Sequences'] / 4
raw_read_counts_nsx_ds['Platform'] = 'NovaSeqX'
raw_read_counts_nsx_ds['Protocol'] = 'Double'
assert set(ns6_ds_multiqc_all.library_id) == set(raw_read_counts_nsx_ds.library_id)
raw_read_counts_nsx_ds = raw_read_counts_nsx_ds.drop(columns=['File', 'Read ID'])
raw_read_counts_nsx_ds = raw_read_counts_nsx_ds.reindex(columns=list(ns6_multiqc_data.columns))

all_multiqc_data = pd.concat([ns6_multiqc_data, raw_read_counts_nsx_ds])

assert (all_multiqc_data[((all_multiqc_data['Lane'] == 'L005') 
     & (all_multiqc_data['Read Type'].isin(['R1', 'R2'])) 
     & (all_multiqc_data['Protocol'] == 'Double')
     & (all_multiqc_data['Platform'] == 'NovaSeqX'))]["fastqc_raw__Total Sequences"] 
        == raw_read_counts_nsx_ds["fastqc_raw__Total Sequences"]).all()

all_multiqc_data['Control'] = all_multiqc_data['fastq_id'].apply(lambda x: 'ExrNTC' if 'ExrNTC' in x else 'Control' if 'Control' in x else None)

data_ns6 = all_multiqc_data[all_multiqc_data["Platform"] == "NovaSeq6"]
data_nsX = all_multiqc_data[all_multiqc_data["Platform"] == "NovaSeqX"]
assert set(data_ns6[data_ns6['Protocol'] == 'Double'].library_id) == set(data_nsX.library_id)

C:\Users\glj523\AppData\Local\Temp\ipykernel_19356\3355779702.py:1: DtypeWarning: Columns (36) have mixed types. Specify dtype option on import or set low_memory=False.
  all_multiqc_data_old = pd.read_csv(r'C:\Users\glj523\OneDrive - University of Copenhagen\data\illumina_analysis\data.csv')


Read counts with controls (Lane 4 has 1 more control than all the other lanes so that control is not included)

In [208]:
import plotly.graph_objects as go

def scatter_plot_plotly_log_range_identity_line(full_dataset, protocol, value_column, lane):
    if protocol == "Single":
        filter = full_dataset["Protocol"] == "Single"
    
    elif protocol == "Double":
        filter = full_dataset["Protocol"] == "Double"
    
    elif protocol == "Both":
        filter = (full_dataset["Protocol"] == "Single") | (full_dataset["Protocol"] == "Double")
        
    else:
        raise Exception("Invalid protocol")
    
    full_dataset = full_dataset[filter]
    
    if lane != 'L004':  
        data_ns6 = full_dataset[(full_dataset["Platform"] == "NovaSeq6") & (full_dataset["Read Type"] == "R1") 
                                        & (full_dataset["Lane"] == lane)
                                        & (full_dataset["library_id"] != 'LV7009026461')]
        data_nsX = full_dataset[(full_dataset["Platform"] == "NovaSeqX") & (full_dataset["Read Type"] == "R1") & (full_dataset["Lane"] == "L005")
                                & (full_dataset["library_id"] != 'LV7009026461')]
    else:
        data_ns6 = full_dataset[(full_dataset["Platform"] == "NovaSeq6") & (full_dataset["Read Type"] == "R1") & (full_dataset["Lane"] == lane)]
        data_nsX = full_dataset[(full_dataset["Platform"] == "NovaSeqX") & (full_dataset["Read Type"] == "R1") & (full_dataset["Lane"] == "L005")]
    
    # Get the x and y values
    x = data_nsX[["library_id", value_column]].groupby("library_id").sum().reset_index().sort_values("library_id")[value_column]
    y = data_ns6[["library_id", value_column]].groupby("library_id").sum().reset_index().sort_values("library_id")[value_column]

    # Filter red points (where Control is not NaN)
    red_points = full_dataset[~full_dataset["Control"].isna()]
    
    if lane != 'L004':  
        data_ns6_red = red_points[(red_points["Platform"] == "NovaSeq6") & (red_points["Read Type"] == "R1") 
                                        & (red_points["Lane"] == lane)
                                        & (red_points["library_id"] != 'LV7009026461')]
        data_nsX_red = red_points[(red_points["Platform"] == "NovaSeqX") & (red_points["Read Type"] == "R1") & (red_points["Lane"] == "L005")
                                  & (red_points["library_id"] != 'LV7009026461')]
    else:
        data_ns6_red = red_points[(red_points["Platform"] == "NovaSeq6") & (red_points["Read Type"] == "R1") & (red_points["Lane"] == lane)]
        data_nsX_red = red_points[(red_points["Platform"] == "NovaSeqX") & (red_points["Read Type"] == "R1") & (red_points["Lane"] == "L005")]

    x_red = data_nsX_red[["library_id", value_column]].groupby("library_id").sum().reset_index().sort_values("library_id")[value_column]
    y_red = data_ns6_red[["library_id", value_column]].groupby("library_id").sum().reset_index().sort_values("library_id")[value_column]
    
    # Create Plotly figure
    fig = go.Figure()

    # Add blue points for non-Red points (Library)
    fig.add_trace(
        go.Scatter(
            x=y,
            y=x,
            mode='markers',
            marker=dict(color='blue'),
            name='Library'
        )
    )

    # Add red points for Control rows
    fig.add_trace(
        go.Scatter(
            x=y_red,
            y=x_red,
            mode='markers',
            marker=dict(color='red'),
            name='Control'
        )
    )

    # Add identity line (y = x) 
    # Create a line for the identity line. Use the minimum and maximum values for x and y
    fig.add_trace(
        go.Scatter(
            x=[0, max(x.max(), y.max())],
            y=[0, max(x.max(), y.max())],
            mode='lines',
            line=dict(color='black', dash='dash'),
            name="y=x"
        )
    )

    # Customize the layout to use log scale and set custom range for both axes
    fig.update_layout(
        title='R1 Read Counts',
        xaxis_title="Seq6000",
        yaxis_title="SeqX",
        showlegend=True,
        width=800,
        height=600,
        xaxis=dict(
            type='log', 
            range=[4, 8],  # Set the range for the x-axis (log scale)
        ),
        yaxis=dict(
            type='log', 
            range=[4, 8],  # Set the range for the y-axis (log scale)
        )
    )

    # Show the plot
    fig.show()
    
    
for lane in ['L001', 'L002', 'L003', 'L004']:

    # Example usage
    scatter_plot_plotly_log_range_identity_line(all_multiqc_data, "Double", "fastqc_raw__Total Sequences", lane=lane)


Zoomed in (Lane 4 has 1 more control than all the other lanes so that control is not included)

In [209]:
import plotly.graph_objects as go

def scatter_plot_plotly_controls_only_with_identity_line_combined(full_dataset, protocol, value_column):
    # Filter the dataset based on the protocol
    if protocol == "Single":
        filter = full_dataset["Protocol"] == "Single"
    elif protocol == "Double":
        filter = full_dataset["Protocol"] == "Double"
    elif protocol == "Both":
        filter = (full_dataset["Protocol"] == "Single") | (full_dataset["Protocol"] == "Double")
    else:
        raise Exception("Invalid protocol")
    
    full_dataset = full_dataset[filter]
    
    # Filter the dataset to include only rows where Control is not NaN
    control_data = full_dataset[~full_dataset["Control"].isna()]
    
    # Create the Plotly figure
    fig = go.Figure()

    # Loop over each lane and add traces to the same figure
    for lane in ['L001', 'L002', 'L003', 'L004']:
        # Filter data for the current lane
        if lane != 'L004':  
            data_ns6_control = control_data[(control_data["Platform"] == "NovaSeq6") & (control_data["Read Type"] == "R1") 
                                            & (control_data["Lane"] == lane) & (control_data["library_id"] != 'LV7009026461')]
            data_nsX_control = control_data[(control_data["Platform"] == "NovaSeqX") & (control_data["Read Type"] == "R1") 
                                            & (control_data["Lane"] == "L005") & (control_data["library_id"] != 'LV7009026461')]
        else:
            data_ns6_control = control_data[(control_data["Platform"] == "NovaSeq6") & (control_data["Read Type"] == "R1") & (control_data["Lane"] == lane)]
            data_nsX_control = control_data[(control_data["Platform"] == "NovaSeqX") & (control_data["Read Type"] == "R1") & (control_data["Lane"] == "L005")]
        
        # Get the x and y values for Control rows
        x_control = data_nsX_control[["library_id", value_column]].groupby("library_id").sum().reset_index().sort_values("library_id")[value_column]
        y_control = data_ns6_control[["library_id", value_column]].groupby("library_id").sum().reset_index().sort_values("library_id")[value_column]

        # Add Control points for this lane
        control_values = data_nsX_control.groupby("library_id")["Control"].first().reset_index()["Control"]

        fig.add_trace(
            go.Scatter(
                x=y_control,
                y=x_control,
                mode='markers',
                marker=dict(color='red'),
                name=f'Control {lane}',
                text=control_values, 
                hovertemplate=(
                    "Control Type:   %{text}<br>" +  # Display the Control value
                    "Seq6000 value:  %{x}<br>" +  # Display Seq6000 value
                    "SeqX value:     %{y}<br>" +  # Display SeqX value
                    "<extra></extra>"  # Remove extra information
                )
            )
        )

    # Add identity line (y = x) once for all lanes
    fig.add_trace(
        go.Scatter(
            x=[0, max(control_data[value_column].max(), control_data[value_column].max())],  # Identity line range
            y=[0, max(control_data[value_column].max(), control_data[value_column].max())],  # Identity line
            mode='lines',
            line=dict(color='black', dash='dash'),
            name="y=x"
        )
    )

    # Customize the layout
    fig.update_layout(
        title="R1 Read counts - Controls Only",
        xaxis_title="Seq6000",
        yaxis_title="SeqX",
        showlegend=True,
        width=1000,  # Set the width of the plot
        height=600,  # Set the height of the plot
        xaxis=dict(range=[30000, 80000]),  # Set x-axis range
        yaxis=dict(range=[30000, 80000])   # Set y-axis range
    )

    # Show the plot
    fig.show()

# Example usage
scatter_plot_plotly_controls_only_with_identity_line_combined(all_multiqc_data, "Double", "fastqc_raw__Total Sequences")


FASTQ R1 read count comparison

R1 sequences count total:

In [6]:

# Data selection and grouping
group_ns6 = data_ns6[(data_ns6["Protocol"] == "Double") & (data_ns6["Read Type"] == "R1")][["Lane", "fastqc_raw__Total Sequences"]].groupby(["Lane"]).sum()
group_reset_ns6 = group_ns6[group_ns6["fastqc_raw__Total Sequences"] > 0].reset_index()

group_nsX = data_nsX[(data_nsX["Protocol"] == "Double") & (data_nsX["Lane"] == "L005") & (data_nsX["Read Type"] == "R1")][["Lane", "fastqc_raw__Total Sequences"]].groupby(["Lane"]).sum()
group_reset_nsX = group_nsX[group_nsX["fastqc_raw__Total Sequences"] > 0].reset_index()

hover_template = "%{y}<extra></extra>"

# Create a single plot
fig = go.Figure()

# Add NS6 data
fig.add_trace(
    go.Bar(
        x=group_reset_ns6["Lane"],
        y=group_reset_ns6["fastqc_raw__Total Sequences"],
        name="NS6",
        hovertemplate=hover_template
    )
)

# Add NSX data
fig.add_trace(
    go.Bar(
        x=group_reset_nsX["Lane"],
        y=group_reset_nsX["fastqc_raw__Total Sequences"],
        name="NSX",
        hovertemplate=hover_template
    )
)

# Update layout
fig.update_layout(
    height=600, width=800,
    yaxis_title="Total R1 Sequences",
    # yaxis_range=[0, 3e10],
    barmode="group",  # Use grouped bars
    title="R1 Sequence Count Total",
    xaxis_title="Lane",
    legend_title="Dataset"
)

# Show the figure
fig.show()


R1 sequences count difference:

In [7]:
difference = group_reset_ns6['fastqc_raw__Total Sequences'] - group_reset_nsX['fastqc_raw__Total Sequences'].iloc[0]
group_reset_ns6['abs_difference'] = difference

# Create a single plot
fig = go.Figure()


# Add NS6 data
fig.add_trace(
    go.Bar(
        x=group_reset_ns6["Lane"],
        y=group_reset_ns6["abs_difference"],
        name="",
        hovertemplate=hover_template,
        showlegend=False  # Ensure mean appears in the legend
    )
)

mean_difference = group_reset_ns6['abs_difference'].mean()


# Overlay mean value as a dashed line
fig.add_trace(
    go.Scatter(
        x=group_reset_ns6["Lane"],
        y=[mean_difference] * len(group_reset_ns6),  # Mean line
        mode="lines",  # Add markers to the line
        name="Mean",
        line=dict(
            color="black",  # Set the color to re
            dash="dot",
            width=3       # Set line thickness
        ),
        showlegend=True  # Ensure mean appears in the legend
    )
)

# Add an annotation for the mean
fig.add_annotation(
    x=0.5,  # Adjust to position annotation centrally
    y=mean_difference + 1000000,
    xref="paper",
    yref="y",
    text=f"{mean_difference / 1_000_000:.2f}M",
    showarrow=False,
    font=dict(color="black", size=16)
)

# Update layout
fig.update_layout(
    height=600, width=800,
    yaxis_title="Difference",
    # yaxis_range=[0, 3e10],
    barmode="group",  # Use grouped bars
    title="R1 Sequences Count Difference from lane 5",
    xaxis_title="Lane",
    legend_title="Dataset"
)

# Show the figure
fig.show()


R1 sequences count percent difference:

In [8]:
difference = group_reset_ns6['abs_difference'] / group_reset_ns6['fastqc_raw__Total Sequences'] * 100
group_reset_ns6['percent_difference'] = difference

# Create the plot
fig = go.Figure()

# Add NS6 data as bars
fig.add_trace(
    go.Bar(
        x=group_reset_ns6["Lane"],
        y=group_reset_ns6["percent_difference"],
        name="Percent Difference",
        hovertemplate="%{y:.2f}%<extra></extra>",
        showlegend=False
    )
)

mean_difference = group_reset_ns6['percent_difference'].mean()

# Overlay mean value as a dashed line
fig.add_trace(
    go.Scatter(
        x=group_reset_ns6["Lane"],
        y=[mean_difference] * len(group_reset_ns6),  # Mean line
        mode="lines",  # Add markers to the line
        name="Mean",
        line=dict(
            color="black",  # Set the color to re
            dash="dot",
            width=3       # Set line thickness
        ),
        showlegend=True  # Ensure mean appears in the legend
    )
)

# Add an annotation for the mean
fig.add_annotation(
    x=0.5,  # Adjust to position annotation centrally
    y=mean_difference +0.05,
    xref="paper",
    yref="y",
    text=f"{mean_difference:.2f}%",
    showarrow=False,
    font=dict(color="black", size=16)
)

# Update layout
fig.update_layout(
    height=600,
    width=800,
    yaxis_title="Percent Difference",
    title="R1 Sequence Count Percent Difference From Lane 5",
    xaxis_title="Lane",
    legend_title="Dataset"
)

fig.show()